# Encoder scatter plot (interactive)

Interactive Plotly scatter plots for encoder outlier analysis. The general scatter uses a meaningful categorical x-axis (target token when available), encoded value on y, stable label ordering, and hover text with enough wrapped context.

The second example uses the sentiment workflow to show the interactive by-target strip plot, which mirrors `encoder_by_target_strip.png` but lets you inspect individual outliers.

**Requires:** `pip install plotly` (optional dependency)


## Setup

In [ ]:
from gradiend import TextPredictionTrainer, TrainingArguments

## Create data and train

In [ ]:
args = TrainingArguments(
    experiment_dir="runs/examples/encoder_scatter",
    train_batch_size=8,
    eval_steps=100,
    max_steps=1000,
    learning_rate=1e-4,
    use_cache=True,
    fail_on_non_convergence=True,
)

pair = ("white", "black")
trainer = TextPredictionTrainer(
    model="distilbert-base-cased",
    run_id=f"race_{pair[0]}_{pair[1]}",
    data="aieng-lab/gradiend_race_data",
    target_classes=list(pair),
    masked_col="masked",
    eval_neutral_data="aieng-lab/biasneutral",
    args=args,
)

trainer.train()

## Encoder evaluation and interactive scatter

In [ ]:
enc_eval = trainer.evaluate_encoder(max_size=100, return_df=True, plot=False)
enc_df = enc_eval.get("encoder_df")

if enc_df is not None and not enc_df.empty:
    trainer.plot_encoder_scatter(
        encoder_df=enc_df,
        show=True,
        max_points=500,
        color_by="label",
        label_name_mapping={"-1.0": "black", "1.0": "white", "0.0": "neutral"},
        title="Race: encoded values by target",
        height=550,
    )
else:
    print("No encoder data. Ensure training completed and use_cache=False if needed.")


## Sentiment by-target outlier analysis

This mirrors the static by-target strip plot, but each point is hoverable. Use it to inspect target words or examples that land far from their expected class.


In [ ]:
from gradiend.examples.train_sentiment import (
    DEFAULT_DATA_DIR,
    DEFAULT_EXPERIMENT_DIR,
    generate_data,
    train as train_sentiment,
)

training_path = DEFAULT_DATA_DIR / "training.csv"
neutral_path = DEFAULT_DATA_DIR / "neutral.csv"
if not (training_path.is_file() and neutral_path.is_file()):
    training_path, neutral_path = generate_data(output_dir=DEFAULT_DATA_DIR)

sentiment_trainer = train_sentiment(
    training_path=training_path,
    neutral_path=neutral_path,
    experiment_dir=DEFAULT_EXPERIMENT_DIR,
    max_steps=150,
    use_cache=True,
)


In [ ]:
sentiment_eval = sentiment_trainer.evaluate_encoder(
    split="all",
    max_size=None,
    return_df=True,
    plot=False,
    use_cache=True,
    include_other_classes=False,
)
sentiment_df = sentiment_eval["encoder_df"]

sentiment_trainer.plot_encoder_by_target(
    encoder_df=sentiment_df,
    plot_style="strip",
    interactive=True,
    title="Sentiment: target-word outlier analysis",
    height=600,
    show=True,
)
